# EmpowerLens — D21 CBT-Bench (CBT-CD) transfer on the D12 winner (Kaggle GPU)

Re-runs Izza's 4 Oct transfer evaluation (`results_cbt_transfer/kaggle-runner-transfer-eval.ipynb`) on the
**D12 winner**: MentalRoBERTa (`mental/mental-roberta-base`) **multilabel**, D12 recipe
(lr 3e-5, 5 epochs, weight decay 0.1, max length 512, batch 16, `data/splits`), seeds **42, 1337, 2024**.

- CBT-CD is loaded exactly as on 4 Oct: `load_dataset("Psychotherapy-LLM/CBT-Bench", "distortions_test", split="train")`,
  labels mapped with the same `CBT_LABEL_CANON` (copied verbatim below), 146 items, 0 dropped.
- Predictions use the **per-type thresholds fitted on val during training** (`meta.json["thresholds"]`,
  also `thresholds.json`), applied with `src.evaluate.predictions_from_logits` — headline rule `cap2`
  (at most 2 labels, the `src.evaluate` default); `nocap` is written too. The 4 Oct run used a fixed 0.5.
- Inputs: `ori_text` (headline, same as 4 Oct) and `situation + thoughts` (CBT-Bench's rewritten fields).
- **Overlap warning:** CBT-CD's `ori_text` posts are rows of our own `Annotated_data.csv` (same Id_Number);
  110 of 146 are in our train split. Every metric is therefore also written for the `not_in_train` subset
  (Id_Number not in `data/splits/train.csv`; test.csv is not read here).
- Per seed: `cbt_predictions_<seed>.csv` (item level) and `cbt_per_class_<seed>.csv`
  (input, decision, subset, label, precision, recall, f1, support; label also `macro` / `micro`), plus
  `meta_<seed>.json`. The checkpoint is deleted after each seed. About 25–35 min per seed on a T4.

**Before running:** the D21 files (`src/cbt_transfer_report.py`, this notebook) must be pushed to `lumia-space`.
Accelerator **GPU (T4 or P100)**, **Internet: On**, Kaggle secret **`HF_TOKEN`** (`mental/mental-roberta-base`
is gated; CBT-Bench itself is public, the 4 Oct notebook logged in anyway).

**Resumable:** a seed is skipped if both of its CSVs are already in `/kaggle/working/d21/`.
**After it finishes:** download `d21_cbt.zip`, copy the contents of `d21/` into `results_cbt_transfer/d21/`
in the repo, then run `venv\Scripts\python.exe -m src.cbt_transfer_report` to regenerate
`results_cbt_transfer/transfer_evaluation_final.md`. Weights never go into git.

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp (not saved with the output), install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt datasets
!ls src/cbt_transfer_report.py data/splits/train.csv && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. Shared config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.data import DISTORTIONS
from src.train_transformer import encode_texts
from src.evaluate import predict_logits, predictions_from_logits
from src.cbt_transfer_report import PER_CLASS_COLS, per_class_rows, to_matrix

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEEDS  = (42, 1337, 2024)
OUT    = Path("/kaggle/working/d21")
CK     = Path("/kaggle/temp/ckpt_d21")
RECIPE = (f"--model {MODEL} --lr 3e-5 --epochs 5 --weight-decay 0.1 "
          f"--max-length 512 --batch-size 16 --splits data/splits")
DECISIONS = {"cap2": 2, "nocap": 0}   # max_labels passed to src.evaluate.predictions_from_logits
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def meta(ck):
    return json.loads((Path(ck) / "meta.json").read_text())

def score_checkpoint(ck, items):
    # Item-level predictions + long per-class table for every (input, decision, subset).
    m = meta(ck)
    thr = m["thresholds"]
    assert thr is not None and len(thr) == len(DISTORTIONS), f"no per-type thresholds in {ck}/meta.json"
    assert json.loads((Path(ck) / "thresholds.json").read_text()) == thr
    tok = AutoTokenizer.from_pretrained(str(ck))
    model = AutoModelForSequenceClassification.from_pretrained(str(ck)).to(DEVICE)
    y_true = to_matrix(items["internal_labels"].tolist())
    pred_rows, class_rows = [], []
    for inp in ("ori_text", "situation_thoughts"):
        enc, trunc = encode_texts(items[inp].astype(str), tok, m["max_length"], m["truncation"], m["head_keep"])
        logits = predict_logits(model, enc, tok.pad_token_id, DEVICE, 32)
        probs = 1 / (1 + np.exp(-logits))
        print(f"  {inp}: truncated {trunc:.1%}")
        for dec, cap in DECISIONS.items():
            y_pred = predictions_from_logits(logits, "multilabel", thr, max_labels=cap)
            for i, (_, it) in enumerate(items.iterrows()):
                pred_rows.append({
                    "id": it["id"], "in_train": bool(it["in_train"]), "input": inp, "decision": dec,
                    "gold": "|".join(it["internal_labels"]),
                    "pred": "|".join(l for l, v in zip(DISTORTIONS, y_pred[i]) if v),
                    **{f"p_{l}": round(float(p), 4) for l, p in zip(DISTORTIONS, probs[i])},
                })
            for sub, mask in (("all", np.ones(len(items), bool)), ("not_in_train", ~items["in_train"].to_numpy())):
                for r in per_class_rows(y_true[mask], y_pred[mask]):
                    class_rows.append({"input": inp, "decision": dec, "subset": sub, **r})
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(pred_rows), pd.DataFrame(class_rows, columns=PER_CLASS_COLS), thr

print("config ok:", MODEL, SEEDS, OUT, DEVICE)

In [ ]:
# 3. Load CBT-CD exactly as the 4 Oct notebook did (results_cbt_transfer/kaggle-runner-transfer-eval.ipynb, cell 4).
from datasets import load_dataset
from src.cbt_transfer_report import CBT_LABEL_CANON as REPORT_CANON

print("Acquiring CBT-Bench (CBT-CD split) from Hugging Face...")
hf_dataset = load_dataset("Psychotherapy-LLM/CBT-Bench", "distortions_test", split="train")
df_external = hf_dataset.to_pandas()

# --- verbatim from the 4 Oct notebook, cell 4 ---
CBT_LABEL_CANON = {
    'all-or-nothing thinking': 'all_or_nothing',
    'magnification': 'magnification',
    'mind reading': 'mind_reading',
    'overgeneralization': 'overgeneralization',
    'personalization': 'personalization',
    'emotional reasoning': 'emotional_reasoning',
    'mental filter': 'mental_filter',
    'should statements': 'should_statements',
    'labeling': 'labeling',
    'fortune-telling': 'fortune_telling'
}

def strict_canon_map(label_array):
    mapped = []
    for label in label_array:
        key = str(label).strip()
        if key not in CBT_LABEL_CANON:
            raise ValueError(f"Unmapped external label {label!r}. No fuzzy matching allowed.")
        mapped.append(CBT_LABEL_CANON[key])
    return mapped

df_external['internal_labels'] = df_external['distortions'].apply(strict_canon_map)

original_count = len(df_external)
df_filtered = df_external[df_external['internal_labels'].map(len) > 0].copy()
dropped_count = original_count - len(df_filtered)
# --- end verbatim ---

assert CBT_LABEL_CANON == REPORT_CANON, "notebook and src/cbt_transfer_report.py mappings differ"
assert (original_count, dropped_count) == (146, 0), (original_count, dropped_count)

items = df_filtered.reset_index(drop=True)
items["situation_thoughts"] = items["situation"].fillna("").str.strip() + " " + items["thoughts"].fillna("").str.strip()
# Overlap check: CBT-CD ids are Annotated_data Id_Numbers. Only train.csv is read (test.csv is evaluate.py's).
train_ids = set(pd.read_csv("data/splits/train.csv", encoding="utf-8-sig")["Id_Number"].astype(int))
items["in_train"] = items["id"].astype(int).isin(train_ids)
print(f"Total loaded: {original_count} | mapped: {len(items)} | dropped: {dropped_count}")
print(f"in our train split: {int(items.in_train.sum())} | not in train: {int((~items.in_train).sum())}")
print("gold labels per class:", to_matrix(items["internal_labels"].tolist()).sum(0).tolist())

In [ ]:
# 4. Smoke test (~3 min): tiny multilabel run, then score CBT-CD with it. Writes nothing to OUT.
sh(f"python -m src.train_transformer --task multilabel --model {MODEL} --seed 42 --smoke --max-length 128 "
   f"--out /kaggle/temp/smoke")
smoke_ck = next(Path("/kaggle/temp/smoke").glob("multilabel_*_smoke"))
p, c, thr = score_checkpoint(smoke_ck, items)
assert len(p) == len(items) * 2 * len(DECISIONS) and set(c["subset"]) == {"all", "not_in_train"}
print("smoke thresholds", thr)
print("SMOKE OK")
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True)

In [ ]:
# 5. Three seeds: train (D12 recipe) -> val thresholds from meta.json -> score CBT-CD -> CSVs -> delete checkpoint.
for seed in SEEDS:
    pred_csv, cls_csv = OUT / f"cbt_predictions_{seed}.csv", OUT / f"cbt_per_class_{seed}.csv"
    if pred_csv.exists() and cls_csv.exists():
        print(f"seed {seed}: already done, skipping"); continue
    name = f"multilabel_{TAG}_{seed}"
    sh(f"python -m src.train_transformer --task multilabel {RECIPE} --seed {seed} --out {CK}")
    m = meta(CK / name)
    assert m["splits"] == "data/splits" and m["model"] == MODEL and m["seed"] == seed
    print(f"seed {seed}: val macro-F1 {m['val_metrics']['eval_macro_f1']:.3f}, best epoch {m['best_epoch']}")
    preds, per_class, thr = score_checkpoint(CK / name, items)
    preds.to_csv(pred_csv, index=False, encoding="utf-8")
    per_class.to_csv(cls_csv, index=False, encoding="utf-8")
    (OUT / f"meta_{seed}.json").write_text(json.dumps(m, indent=2), encoding="utf-8")
    head = per_class[(per_class.input == "ori_text") & (per_class.decision == "cap2") & (per_class.subset == "all")]
    print(head.set_index("label").loc[["macro", "micro"], ["precision", "recall", "f1", "support"]].round(3))
    shutil.rmtree(CK / name, ignore_errors=True)   # weights never kept
shutil.rmtree(CK, ignore_errors=True)

In [ ]:
# 6. Report (mean +/- std over seeds, 4 Oct column) + zip for download.
sh(f"python -m src.cbt_transfer_report --dir {OUT} --out {OUT / 'transfer_evaluation_final.md'}")
print((OUT / "transfer_evaluation_final.md").read_text(encoding="utf-8"))
!cd /kaggle/working && rm -f d21_cbt.zip && zip -qr d21_cbt.zip d21
!ls -la /kaggle/working/d21 && du -sh /kaggle/working/d21_cbt.zip